# RAG Corpus Preparation
Builds the RAG knowledge base for the GNN + RAG threat detection project.

| Step | What it does | Output |
|---|---|---|
| 1. parse_attack | MITRE ATT&CK STIX → technique lookup table (+ mitigations) | `mitre_lookup.csv`, `mitre_mitigations.csv` |
| 2. parse_cve | cvelistV5 JSON → one chunk per CVE | `cve_chunks.jsonl` |
| 3. parse_cti | CTI-HAL annotations → one chunk per annotated span | `cti_chunks.jsonl` |
| 4. build_index | Embed chunks → Chroma (collections `cve`, `cti`) | `chroma_db/` |
| 5. retrieve(event) | MITRE lookup → CVE (product / technique-gated) → CTI with fallback | demo output |
| 6. Golden dataset | Baseline retrieval evaluation on CTI-HAL labels | `golden_cti.jsonl`, `eval_cti_baseline.csv` |
| 7. UWF integration | ATT&CK procedure examples, CTID KEV→ATT&CK CVE mapping, `edge_to_event()`, coverage of UWF techniques, UWF golden queries | `procedure_chunks.jsonl`, `uwf_coverage.csv`, `golden_cti_uwf.jsonl`, `eval_cti_uwf.csv` |

Shared chunk schema (steps 2–3, 7): `{"id": str, "text": str, "metadata": {...}}`.
Metadata values are scalars (str/int/float), except `technique_ids` (list of str), which needs chromadb >= 1.5.

## 0. Setup

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install "chromadb>=1.5" sentence-transformers

In [ ]:
!ls /content/drive/Shareddrives/

In [ ]:
!echo "== 298B 根目录 =="; ls -la "/content/drive/Shareddrives/298B/"
!echo "== Phase B Preprocessing =="; ls -la "/content/drive/Shareddrives/298B/Phase B Preprocessing/"
!echo "== RAG Corpus =="; ls -la "/content/drive/Shareddrives/298A Group/Data/RAG Corpus/"
!echo "== RAG processed =="; ls -la "/content/drive/Shareddrives/298A Group/Data/RAG Corpus/processed/"
!echo "== 搜索 uwf_graph_v2 / edges_win =="; find /content/drive/Shareddrives -maxdepth 4 \( -name "uwf_graph_v2" -o -name "edges_win_*.parquet" \) 2>/dev/null

In [ ]:
import zipfile, os
ZIP  = "/content/drive/Shareddrives/298B/Phase B Preprocessing/298B-20261001T192511Z-1-001.zip"
DEST = "/content/drive/Shareddrives/298B/Phase B Preprocessing/uwf_graph_v2"

z = zipfile.ZipFile(ZIP)
files = [n for n in z.namelist() if not n.endswith("/")]
print(len(files), "files; 非 raw 数据文件：")
print([n for n in files if "uwf_raw" not in n])

names = [n for n in files if n.endswith(("edges_win_15min.parquet", "node_static.csv"))]
print("\n要取的文件：", names)
if names:
    os.makedirs(DEST, exist_ok=True)
    for n in names:
        with open(f"{DEST}/{os.path.basename(n)}", "wb") as f:
            f.write(z.read(n))
    !ls -lh "{DEST}"

In [ ]:
import json, re, glob, shutil, subprocess, time, difflib
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
import pandas as pd

# ---------------- Config (edit here) ----------------
DRIVE_ROOT = os.environ.get("RAG_DRIVE_ROOT", "/content/drive/Shareddrives/298A Group/Data/RAG Corpus")
CVE_DIR = f"{DRIVE_ROOT}/CVE"                              # CVE/<year>/<nxxx>/CVE-*.json
ATTACK_JSON = os.environ.get("RAG_ATTACK_JSON", f"{DRIVE_ROOT}/enterprise-attack.json")  # adjust if stored elsewhere
OUT_DIR = os.environ.get("RAG_OUT_DIR", f"{DRIVE_ROOT}/processed")   # outputs saved back to Drive
LOCAL_DIR = os.environ.get("RAG_LOCAL_DIR", "/content/rag_work")    # fast local disk for clone / Chroma build

CVE_YEARS = [2021, 2022, 2023, 2024, 2025, 2026]   # match UWF dataset range
CVE_NETWORK_ONLY = True          # keep only CVEs mentioning network services (HTTP, SMB, SSH, ...)
CVE_MAX_RECORDS = None           # e.g. 5000 for a quick test; None = no cap

CTI_HAL_REPO = "https://github.com/dessertlab/CTI-HAL.git"
CTI_APT29_ANNOTATOR = "annotator L"   # APT29 was annotated twice (L and S) on the same reports; keep one

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # alternative: "BAAI/bge-small-en-v1.5"
# ----------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)


def write_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

## 1. parse_attack – MITRE lookup table
- Keeps `attack-pattern` objects that are not revoked or deprecated (techniques + sub-techniques).
- Tactic IDs/names come from `x-mitre-tactic` objects (matched on the kill-chain phase shortname).
- Mitigations come from `relationship` objects of type `mitigates` (course-of-action → attack-pattern).

In [ ]:
def _attack_id(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id"), ref.get("url")
    return None, None


def _active(obj):
    return not obj.get("revoked", False) and not obj.get("x_mitre_deprecated", False)


def parse_attack(path):
    with open(path, encoding="utf-8") as f:
        objects = json.load(f)["objects"]

    # tactic shortname -> (TA id, name)
    tactics = {}
    for o in objects:
        if o.get("type") == "x-mitre-tactic" and _active(o):
            ta_id, _ = _attack_id(o)
            tactics[o["x_mitre_shortname"]] = (ta_id, o["name"])

    techniques, stix_to_tid = [], {}
    for o in objects:
        if o.get("type") != "attack-pattern" or not _active(o):
            continue
        tid, url = _attack_id(o)
        if not tid:
            continue
        stix_to_tid[o["id"]] = tid
        phases = [p["phase_name"] for p in o.get("kill_chain_phases", [])
                  if p.get("kill_chain_name") == "mitre-attack"]
        techniques.append({
            "technique_id": tid,
            "name": o.get("name", ""),
            "is_subtechnique": bool(o.get("x_mitre_is_subtechnique", False)),
            "parent_id": tid.split(".")[0],
            "tactic_ids": ", ".join(tactics.get(p, ("", ""))[0] for p in phases),
            "tactics": ", ".join(tactics.get(p, ("", p))[1] for p in phases),
            "platforms": ", ".join(o.get("x_mitre_platforms", [])),
            "description": o.get("description", ""),
            "detection": o.get("x_mitre_detection", ""),   # may be empty in newer ATT&CK versions
            "url": url,
        })

    mitigations_by_stix = {}
    for o in objects:
        if o.get("type") == "course-of-action" and _active(o):
            mid, _ = _attack_id(o)
            mitigations_by_stix[o["id"]] = (mid, o.get("name", ""))

    mit_rows = []
    for o in objects:
        if (o.get("type") == "relationship" and o.get("relationship_type") == "mitigates"
                and _active(o) and o.get("source_ref") in mitigations_by_stix
                and o.get("target_ref") in stix_to_tid):
            mid, mname = mitigations_by_stix[o["source_ref"]]
            mit_rows.append({
                "technique_id": stix_to_tid[o["target_ref"]],
                "mitigation_id": mid,
                "mitigation_name": mname,
                "how": o.get("description", ""),
            })

    lookup = pd.DataFrame(techniques).sort_values("technique_id").reset_index(drop=True)
    mitigations = pd.DataFrame(mit_rows, columns=["technique_id", "mitigation_id", "mitigation_name", "how"])
    summary = (mitigations.groupby("technique_id")
               .apply(lambda g: "; ".join(f"{a} {b}" for a, b in zip(g.mitigation_id, g.mitigation_name)),
                      include_groups=False)
               .rename("mitigations"))
    lookup = lookup.merge(summary, on="technique_id", how="left").fillna({"mitigations": ""})
    return lookup, mitigations


mitre_lookup, mitre_mitigations = parse_attack(ATTACK_JSON)
mitre_lookup.to_csv(f"{OUT_DIR}/mitre_lookup.csv", index=False)
mitre_mitigations.to_csv(f"{OUT_DIR}/mitre_mitigations.csv", index=False)
print(f"{len(mitre_lookup)} techniques ({mitre_lookup.is_subtechnique.sum()} sub-techniques), "
      f"{len(mitre_mitigations)} technique–mitigation links")
mitre_lookup.head()

In [ ]:
MITRE_INDEX = mitre_lookup.set_index("technique_id")


def lookup_technique(tid):
    """Exact MITRE lookup used at query time, e.g. lookup_technique('T1190')."""
    if tid not in MITRE_INDEX.index:
        return None
    return MITRE_INDEX.loc[tid].to_dict() | {"technique_id": tid}


lookup_technique("T1190")

## 2. parse_cve – CVE chunking
- Input layout: `CVE/<year>/<nxxx>/CVE-<year>-<n>.json` (cvelistV5, CVE JSON 5.x).
- Skips non-PUBLISHED records (e.g. REJECTED).
- Chunk text = CVE ID + description + affected products + CWE.
- CWE and CVSS are taken from the CNA container first, then ADP containers (e.g. CISA-ADP).
- Files are read with a thread pool because reading many small files from a Drive mount is slow.

In [ ]:
NETWORK_PATTERN = re.compile(
    r"\b(https?|web ?server|web application|apache|nginx|iis|tomcat|"
    r"smb|samba|cifs|ssh|openssh|rdp|remote desktop|ftp|sftp|dns|bind9|smtp|"
    r"ldap|snmp|telnet|kerberos|ntp|dhcp|ssl|tls|openssl|vpn|rpc|netbios)\b",
    re.IGNORECASE,
)
CVSS_KEYS = ["cvssV4_0", "cvssV3_1", "cvssV3_0", "cvssV2_0"]   # preference order


def _en_text(items, key="value"):
    for d in items or []:
        if str(d.get("lang", "")).lower().startswith("en") and d.get(key):
            return d[key].strip()
    return ""


def _cvss(container):
    for m in container.get("metrics", []) or []:
        for k in CVSS_KEYS:
            if k in m and "baseScore" in m[k]:
                return float(m[k]["baseScore"]), m[k].get("baseSeverity", ""), k
    return None


def _cwes(container):
    out = []
    for pt in container.get("problemTypes", []) or []:
        for d in pt.get("descriptions", []) or []:
            if d.get("cweId"):
                out.append((d["cweId"], d.get("description", "")))
    return out


def _versions(aff):
    out = []
    for v in aff.get("versions", []) or []:
        if v.get("status") != "affected":
            continue
        ver = v.get("version", "")
        if v.get("lessThan"):
            out.append(f"{ver} to <{v['lessThan']}" if ver not in ("", "0", "*", "n/a") else f"<{v['lessThan']}")
        elif v.get("lessThanOrEqual"):
            out.append(f"{ver} to <={v['lessThanOrEqual']}" if ver not in ("", "0", "*", "n/a") else f"<={v['lessThanOrEqual']}")
        elif ver and ver.lower() != "n/a":
            out.append(ver)
    return out


def parse_cve_record(rec):
    meta = rec.get("cveMetadata", {})
    if meta.get("state") != "PUBLISHED":
        return None
    cna = rec.get("containers", {}).get("cna", {})
    adps = rec.get("containers", {}).get("adp", []) or []

    desc = _en_text(cna.get("descriptions"))
    if not desc:
        return None

    vendors, products, versions, affected_lines = [], [], [], []
    for aff in (cna.get("affected") or [])[:10]:
        vendor = (aff.get("vendor") or "").strip()
        product = (aff.get("product") or "").strip()
        if vendor.lower() == "n/a": vendor = ""
        if product.lower() == "n/a": product = ""
        vers = _versions(aff)
        if vendor and vendor not in vendors: vendors.append(vendor)
        if product and product not in products: products.append(product)
        versions += vers
        if vendor or product:
            affected_lines.append(f"{vendor} {product} {', '.join(vers[:5])}".strip())

    cwes = _cwes(cna) or [c for a in adps for c in _cwes(a)]
    cvss = _cvss(cna) or next((c for a in adps if (c := _cvss(a))), None)

    cwe_text = "; ".join(f"{cid} {name.replace(cid, '').strip(' :')}".strip() for cid, name in cwes)
    text = f"{meta['cveId']}: {desc}"
    if affected_lines:
        text += "\nAffected: " + "; ".join(affected_lines[:5])
    if cwe_text:
        text += "\nWeakness: " + cwe_text

    published = (meta.get("datePublished") or "")[:10]
    return {
        "id": meta["cveId"],
        "text": text,
        "metadata": {
            "source": "cve",
            "cve_id": meta["cveId"],
            "vendor": "; ".join(vendors),
            "product": "; ".join(products),
            "version": "; ".join(versions)[:500],
            "cwe": ", ".join(cid for cid, _ in cwes),
            "cvss": cvss[0] if cvss else -1.0,          # -1 = no score available
            "severity": cvss[1] if cvss else "",
            "cvss_version": cvss[2] if cvss else "",
            "published": published,
            "year": int(published[:4]) if published else -1,
        },
    }


def _load_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        return {"_error": f"{path}: {e}"}


def parse_cve(cve_dir, years, network_only=True, max_records=None, workers=32):
    paths = []
    for y in years:
        paths += glob.glob(f"{cve_dir}/{y}/**/CVE-*.json", recursive=True)
    print(f"Found {len(paths):,} CVE files for years {years}")

    chunks, stats = [], Counter()
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for i, rec in enumerate(pool.map(_load_json, paths), 1):
            if "_error" in rec:
                stats["read_error"] += 1
                continue
            chunk = parse_cve_record(rec)
            if chunk is None:
                stats["not_published_or_no_description"] += 1
                continue
            if network_only and not NETWORK_PATTERN.search(chunk["text"]):
                stats["filtered_non_network"] += 1
                continue
            chunks.append(chunk)
            if max_records and len(chunks) >= max_records:
                break
            if i % 10000 == 0:
                print(f"  {i:,} files read ({time.time() - t0:.0f}s)")
    stats["kept"] = len(chunks)
    print(dict(stats), f"| {time.time() - t0:.0f}s")
    return chunks


from collections import defaultdict

def cve_year_path(y):
    return f"{OUT_DIR}/cve_chunks_{y}.jsonl"

# One-time: split the old combined cache (2024–2026) into per-year files, keyed by the year in the CVE ID
old_cache = f"{OUT_DIR}/cve_chunks.jsonl"
if os.path.exists(old_cache) and not any(os.path.exists(cve_year_path(y)) for y in CVE_YEARS):
    by_year = defaultdict(list)
    for c in read_jsonl(old_cache):
        by_year[int(c["id"].split("-")[1])].append(c)
    for y, part in by_year.items():
        write_jsonl(part, cve_year_path(y))
    print("Split old cache into per-year files:", {y: len(p) for y, p in sorted(by_year.items())})

REBUILD_CVE = False   # True = rescan every year in CVE_YEARS, ignoring caches
cve_chunks = []
for y in CVE_YEARS:
    path = cve_year_path(y)
    if os.path.exists(path) and not REBUILD_CVE:
        part = read_jsonl(path)
        print(f"{y}: loaded {len(part):,} cached chunks")
    else:
        part = parse_cve(CVE_DIR, [y], CVE_NETWORK_ONLY, CVE_MAX_RECORDS)
        write_jsonl(part, path)   # saved per year, so a disconnect keeps finished years
    cve_chunks += part

write_jsonl(cve_chunks, old_cache)   # combined file read by section 4
print(f"Total: {len(cve_chunks):,} CVE chunks for {CVE_YEARS[0]}–{CVE_YEARS[-1]}")
print(json.dumps(cve_chunks[0], indent=2)[:1500] if cve_chunks else "No CVE chunks")

In [ ]:
cve_df = pd.DataFrame([c["metadata"] for c in cve_chunks])
if len(cve_df):
    print("Records per year:\n", cve_df.year.value_counts().sort_index().to_string())
    print("\nWith product:", (cve_df["product"] != "").mean().round(3),
          "| with CWE:", (cve_df.cwe != "").mean().round(3),
          "| with CVSS:", (cve_df.cvss >= 0).mean().round(3))
    print("\nTop vendors:\n", cve_df.vendor.value_counts().head(10).to_string())

## 3. parse_cti – CTI-HAL chunking
- `data/<group>/*.json` holds human-annotated text spans (`context`) with a MITRE technique label.
- One chunk per unique span. If the same span is labeled with several techniques, the labels are merged into `technique_ids`.
- Entries without a valid technique ID or with fewer than 4 words (e.g. just a tool name) are dropped; IDs missing the "T" prefix are fixed.
- Chunk text is the raw report span only — the technique label is kept out of the text
  so it can be used as ground truth for retrieval evaluation.
- Report title/URL come from the tables in the CTI-HAL README.

In [ ]:
cti_dir = f"{LOCAL_DIR}/CTI-HAL"
if not os.path.exists(cti_dir):
    subprocess.run(["git", "clone", "-q", "--depth", "1", CTI_HAL_REPO, cti_dir], check=True)


def parse_readme_reports(readme_path):
    """(group, identifier) -> (title, url) from the README tables."""
    reports, group = {}, None
    row = re.compile(r"^\|\s*\[(.+?)\]\((.+?)\)\s*\|\s*(.*?)\s*\|\s*(.+?)\s*\|")
    with open(readme_path, encoding="utf-8") as f:
        for line in f:
            if line.startswith("## ") and line.strip().endswith("Reports"):
                group = line[3:].replace("Reports", "").strip().lower()
            elif group and (m := row.match(line)):
                reports[(group, m.group(4).strip().upper())] = (m.group(1).strip(), m.group(2).strip())
    return reports


def match_report(reports, group, report_id):
    """File names don't always match README identifiers exactly (case, prefixes, typos)."""
    rid = report_id.upper()
    if (group, rid) in reports:
        return reports[(group, rid)]
    candidates = [k[1] for k in reports if k[0] == group]
    for c in candidates:
        if c.startswith(rid) or rid.startswith(c):
            return reports[(group, c)]
    close = difflib.get_close_matches(rid, candidates, n=1, cutoff=0.8)
    return reports[(group, close[0])] if close else ("", "")


def _norm_tid(t, sub=False):
    if not isinstance(t, str):
        return None
    t = t.strip().upper()
    if re.fullmatch(r"\d{4}(\.\d{3})?", t):
        t = "T" + t
    pattern = r"T\d{4}\.\d{3}" if sub else r"T\d{4}"
    return t if re.fullmatch(pattern, t) else None


def parse_cti(repo_dir, apt29_annotator="annotator L"):
    reports = parse_readme_reports(f"{repo_dir}/README.md")
    chunks, stats, seen = [], Counter(), {}
    for path in sorted(glob.glob(f"{repo_dir}/data/**/*.json", recursive=True)):
        rel = os.path.relpath(path, f"{repo_dir}/data").split(os.sep)
        group = rel[0]
        if group == "apt29" and len(rel) > 2 and rel[1] != apt29_annotator:
            continue
        report_id = os.path.splitext(rel[-1])[0]
        title, url = match_report(reports, group, report_id)
        if not title:
            stats["report_not_in_readme"] += 1

        with open(path, encoding="utf-8") as f:
            entries = json.load(f)
        for i, e in enumerate(entries):
            m = e.get("metadata", {}) or {}
            text = (e.get("context") or "").strip()
            tid = _norm_tid(e.get("technique"))
            if tid is None:
                stats["dropped_no_valid_technique"] += 1
                continue
            if len(text.split()) < 4:
                stats["dropped_too_short"] += 1
                continue
            tactics = [t.title() for t in (m.get("tactic_name") or [])]
            key = (group, report_id, text)
            if key in seen:
                # Same span labeled with another technique -> merge labels into one chunk
                md = seen[key]["metadata"]
                if tid not in md["technique_ids"]:
                    md["technique_ids"].append(tid)
                    stats["merged_extra_label"] += 1
                else:
                    stats["dropped_exact_duplicate"] += 1
                for t in tactics:
                    if t not in md["tactic"]:
                        md["tactic"] += (", " if md["tactic"] else "") + t
                continue
            chunk = {
                "id": f"{group}-{report_id}-{m.get('id') or i}-{i}",
                "text": text,
                "metadata": {
                    "source": "cti-hal",
                    "technique_id": tid,           # primary label
                    "technique_ids": [tid],        # all labels; filter with {"technique_ids": {"$contains": tid}}
                    "technique_name": (m.get("technique_name") or "").title(),
                    "sub_technique": _norm_tid(m.get("sub_technique"), sub=True) or "",
                    "tactic": ", ".join(tactics),
                    "apt_group": group.upper(),
                    "tool": ", ".join(m.get("tool_name") or []),
                    "report": report_id,
                    "report_title": title,
                    "report_url": url,
                    "page": m.get("page_number") if isinstance(m.get("page_number"), int) else -1,
                    "annotation": (m.get("description") or "").strip(),
                },
            }
            seen[key] = chunk
            chunks.append(chunk)
    stats["kept"] = len(chunks)
    print(dict(stats))
    return chunks


cti_chunks = parse_cti(cti_dir, CTI_APT29_ANNOTATOR)
write_jsonl(cti_chunks, f"{OUT_DIR}/cti_chunks.jsonl")
print(json.dumps(cti_chunks[0], indent=2))

In [ ]:
cti_df = pd.DataFrame([c["metadata"] for c in cti_chunks])
labels = cti_df.explode("technique_ids")
print(f"{labels.technique_ids.nunique()} techniques, {cti_df.report.nunique()} reports, "
      f"{(cti_df.technique_ids.str.len() > 1).sum()} chunks with multiple labels")
print("\nChunks per APT group:\n", cti_df.apt_group.value_counts().to_string())
print("\nTop techniques:\n", labels.technique_ids.value_counts().head(10).to_string())
# Coverage check for techniques that occur in UWF (sorted by attack edges, 15-min windows, from the preprocessing notebook)
UWF_TECHNIQUES = ["T1595", "T1110", "T1078", "T1190", "T1046", "T1018", "T1548", "T1592",
                  "T1048", "T1587", "T1589", "T1210", "T1016", "T1071", "T1590", "T1059"]
print("\nCTI-HAL coverage of network techniques:\n",
      labels.technique_ids.value_counts().reindex(UWF_TECHNIQUES, fill_value=0).to_string())

## 4. build_index – Chroma vector store
- Embeds chunk text with sentence-transformers (normalized, cosine distance).
- Two collections: `cve` and `cti`; metadata is stored for filtering, e.g.
  `where={"technique_ids": {"$contains": "T1190"}}` (CTI) or `where={"product": "Apache HTTP Server"}` (CVE).
- Built on local disk (Chroma's SQLite is unreliable on a Drive mount), then copied to Drive.

In [ ]:
import chromadb


def get_embedder(model_name):
    if os.environ.get("RAG_FAKE_EMBED"):          # offline testing only
        import hashlib, numpy as np
        def fake(texts):
            out = []
            for t in texts:
                rng = np.random.default_rng(int(hashlib.md5(t.encode()).hexdigest()[:8], 16))
                v = rng.normal(size=384); out.append((v / np.linalg.norm(v)).tolist())
            return out
        return fake
    import torch
    from sentence_transformers import SentenceTransformer
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Embedding with {model_name} on {device}")
    model = SentenceTransformer(model_name, device=device)
    return lambda texts: model.encode(texts, batch_size=128, normalize_embeddings=True,
                                      show_progress_bar=False).tolist()


def build_collection(client, name, chunks, embed, batch=2000):
    if name in [c.name for c in client.list_collections()]:
        client.delete_collection(name)                # rebuild from scratch on each run
    col = client.create_collection(name, metadata={"hnsw:space": "cosine", "embed_model": EMBED_MODEL})
    t0 = time.time()
    for i in range(0, len(chunks), batch):
        part = chunks[i:i + batch]
        col.add(ids=[c["id"] for c in part],
                documents=[c["text"] for c in part],
                metadatas=[c["metadata"] for c in part],
                embeddings=embed([c["text"] for c in part]))
        print(f"  {name}: {min(i + batch, len(chunks)):,}/{len(chunks):,} ({time.time() - t0:.0f}s)")
    return col


local_chroma = f"{LOCAL_DIR}/chroma_db"
shutil.rmtree(local_chroma, ignore_errors=True)
client = chromadb.PersistentClient(path=local_chroma)
embed = get_embedder(EMBED_MODEL)

cve_col = build_collection(client, "cve", read_jsonl(f"{OUT_DIR}/cve_chunks.jsonl"), embed)
cti_col = build_collection(client, "cti", read_jsonl(f"{OUT_DIR}/cti_chunks.jsonl"), embed)
print({"cve": cve_col.count(), "cti": cti_col.count()})

In [ ]:
# Copy the index to Drive so it can be reused without re-embedding
drive_chroma = f"{OUT_DIR}/chroma_db"
del client                                   # release the SQLite file before copying
shutil.rmtree(drive_chroma, ignore_errors=True)
shutil.copytree(local_chroma, drive_chroma)
print("Saved index to", drive_chroma)

# To reuse later without rebuilding:
#   shutil.copytree(f"{OUT_DIR}/chroma_db", f"{LOCAL_DIR}/chroma_db", dirs_exist_ok=True)
#   client = chromadb.PersistentClient(path=f"{LOCAL_DIR}/chroma_db")

### Sanity check – does retrieval return sensible results?

In [ ]:
client = chromadb.PersistentClient(path=local_chroma)
cve_col, cti_col = client.get_collection("cve"), client.get_collection("cti")


def search(col, query, k=5, where=None):
    res = col.query(query_embeddings=embed([query]), n_results=k, where=where)
    return pd.DataFrame({
        "score": [round(1 - d, 3) for d in res["distances"][0]],   # cosine similarity
        "text": [t[:150] for t in res["documents"][0]],
        **{key: [m.get(key, "") for m in res["metadatas"][0]]
           for key in (["technique_id", "apt_group", "report"] if col.name == "cti" else ["cve_id", "product", "cvss"])},
    })


q = "Exploit public-facing web application over HTTP to gain initial access"
print("CTI, no filter:");            print(search(cti_col, q).to_string())
print("\nCTI, technique_id = T1190:"); print(search(cti_col, q, where={"technique_ids": {"$contains": "T1190"}}).to_string())
print("\nCVE:");                      print(search(cve_col, "Apache HTTP Server path traversal remote code execution").to_string())

## 5. retrieve(event) – retrieval demo
Retrieval flow per detected event (simulated GNN output, or a UWF edge converted with `edge_to_event()` in section 7):
1. **MITRE lookup** – exact match on technique_id (sub-technique falls back to its parent). UWF labels such as
   `Reconnaissance_unspecified` have no technique, so only the tactic is used.
2. **CVE search** – `CVE_MODE` decides when it runs:
   - `product`: only if the event names a product (UWF conn logs have none, so this never fires on UWF);
   - `ctid`: exploit-type techniques only (`CVE_TECHNIQUES`), restricted to CVEs mapped to that technique in CTID's KEV→ATT&CK mappings (section 7.3);
   - `service`: exploit-type techniques only, CVE text must mention the event's service (HTTP, SSH, ...);
   - `auto` (default): product if present → else ctid if a mapping exists → else service; `off`: never.
3. **CTI search** – CTI chunks labeled with the (parent) technique. If fewer than `k_cti` are found and `fallback=True`:
   ATT&CK procedure examples for the same technique (section 7.2), then CTI chunks of other techniques in the same tactic.
   Each row is tagged `evidence_level` = technique / procedure / tactic.

Run sections 0–4 first (or reload the saved index, see the note at the end of section 4).

In [ ]:
# ---------------- Retrieval config (edit here) ----------------
CVE_MODE = "auto"   # "auto" | "product" | "ctid" | "service" | "off"
# Exploit-type techniques: the only ones where a CVE is meaningful without a product name.
# Recon / brute force / valid accounts (T1595, T1110, T1078, T1046, ...) do not exploit a vulnerability.
CVE_TECHNIQUES = {"T1190", "T1210", "T1203", "T1211", "T1212", "T1068"}
SERVICE_KEYWORDS = {"http": ["HTTP"], "ssl": ["TLS", "SSL"], "ssh": ["SSH"], "dns": ["DNS"],
                    "smb": ["SMB"], "rdp": ["RDP", "Remote Desktop"], "ftp": ["FTP"]}
CTID_CVES = globals().get("CTID_CVES", {})      # parent technique -> set of CVE IDs; filled in section 7.3
# --------------------------------------------------------------

SEARCH_COLS = {"cti": ["technique_ids", "apt_group", "report", "page"],
               "procedures": ["technique_ids", "source_name", "source_type"],
               "cve": ["cve_id", "product", "version", "cvss"]}


def search(col, query, k=5, where=None, where_document=None):
    res = col.query(query_embeddings=embed([query]), n_results=k, where=where, where_document=where_document)
    cols = SEARCH_COLS[col.name]
    return pd.DataFrame({
        "score": [round(1 - d, 3) for d in res["distances"][0]],   # cosine similarity
        "text": [t[:160].replace("\n", " ") for t in res["documents"][0]],
        **{c: [m.get(c, "") for m in res["metadatas"][0]] for c in cols},
    })


def _mitre_for(tid):
    return lookup_technique(tid) or lookup_technique(tid.split(".")[0])


def _first_sentence(text):
    text = re.sub(r"\(Citation:[^)]*\)", "", text or "")
    text = re.sub(r"\s+([.,;])", r"\1", text).strip()
    return re.split(r"(?<=\.)\s", text, maxsplit=1)[0]


def _resolve_ids(event):
    """-> (technique_id or None, tactic or None). UWF uses '<Tactic>_unspecified' when the technique is unknown."""
    tid = str(event.get("technique_id") or "").strip()
    tactic = event.get("tactic") or None
    if tid.endswith("_unspecified"):
        tactic, tid = tactic or tid[: -len("_unspecified")], None
    elif not re.fullmatch(r"T\d{4}(\.\d{3})?", tid):
        tid = None
    return tid, tactic


def _parents_for_tactic(tactic):
    """Parent technique IDs whose ATT&CK tactics include `tactic` (e.g. 'Reconnaissance')."""
    hit = mitre_lookup[mitre_lookup.tactics.str.split(", ").apply(lambda ts: tactic in ts)]
    return sorted(set(hit.parent_id))


def _contains_any(field, values):
    conds = [{field: {"$contains": v}} for v in values]
    return conds[0] if len(conds) == 1 else {"$or": conds}


def _cve_mode_for(event, parent, mode):
    if mode == "off":
        return None
    if event.get("product") and mode in ("auto", "product"):
        return "product"
    if mode == "product" or parent not in CVE_TECHNIQUES:
        return None
    if mode in ("auto", "ctid") and CTID_CVES.get(parent):
        return "ctid"
    return "service" if mode in ("auto", "service") else None


def _service_keywords(event):
    kws = []
    for s in [event.get("service"), event.get("service_hint")]:
        kws += SERVICE_KEYWORDS.get(str(s or "").lower(), [])
    return list(dict.fromkeys(kws))


def retrieve(event, k_cve=3, k_cti=5, use_filter=True, cve_mode=None, fallback=True):
    """event: {technique_id, tactic?, service, port, protocol, behavior, product?, version?}"""
    tid, tactic = _resolve_ids(event)
    parent = tid.split(".")[0] if tid else None
    mitre = _mitre_for(tid) if tid else None
    name = mitre["name"] if mitre else (f"{tactic} activity" if tactic else "network activity")
    tactics = [tactic] if tactic else (mitre["tactics"].split(", ") if mitre and mitre["tactics"] else [])
    out = {"event": event, "mitre": mitre, "tactics": tactics, "cve": None, "cve_mode": None, "cti": None}

    # Step 2: CVE
    mode = _cve_mode_for(event, parent, cve_mode or CVE_MODE) if use_filter else None
    out["cve_mode"] = mode
    if mode == "product":
        q = " ".join(str(x) for x in [name, event["product"], event.get("version"), event.get("service"), "vulnerability"] if x)
        out["cve"] = search(cve_col, q, k=k_cve, where_document={"$contains": event["product"]})
    elif mode in ("ctid", "service"):
        q = " ".join(str(x) for x in [name, event.get("service_hint") or event.get("service"), event.get("behavior"), "vulnerability"] if x)
        if mode == "ctid":
            out["cve"] = search(cve_col, q, k=k_cve, where={"cve_id": {"$in": sorted(CTID_CVES[parent])}})
        else:
            kws = _service_keywords(event)
            wd = None if not kws else ({"$contains": kws[0]} if len(kws) == 1 else {"$or": [{"$contains": k} for k in kws]})
            out["cve"] = search(cve_col, q, k=k_cve, where_document=wd)

    # Step 3: CTI (+ fallbacks)
    q = (f"{name}: {event.get('behavior', '')} over {event.get('service', '')}/"
         f"{event.get('protocol', '')} port {event.get('port', '')}")
    if not use_filter:
        out["cti"] = search(cti_col, q, k=k_cti).assign(evidence_level="unfiltered")
        return out
    parts = []
    if parent:
        parts.append(search(cti_col, q, k=k_cti, where={"technique_ids": {"$contains": parent}})
                     .assign(evidence_level="technique"))
    have = sum(len(p) for p in parts)
    proc = globals().get("proc_col")
    if fallback and parent and proc is not None and have < k_cti:
        parts.append(search(proc, q, k=k_cti - have, where={"technique_ids": {"$contains": parent}})
                     .assign(evidence_level="procedure"))
        have = sum(len(p) for p in parts)
    if fallback and tactics and have < k_cti:
        sib = [t for ta in tactics for t in _parents_for_tactic(ta) if t != parent]
        if sib:
            parts.append(search(cti_col, q, k=k_cti - have, where=_contains_any("technique_ids", sorted(set(sib))))
                         .assign(evidence_level="tactic"))
    out["cti"] = (pd.concat(parts, ignore_index=True).drop_duplicates("text").head(k_cti).fillna("") if parts
                  else pd.DataFrame(columns=["score", "text", "evidence_level"]))
    return out


def show(result):
    e, m = result["event"], result["mitre"]
    print(f"EVENT  {e.get('technique_id')} | {e.get('service')}/{e.get('protocol')}:{e.get('port')} | {e.get('behavior')}")
    if m:
        print(f"\n[1] MITRE  {m['technique_id']} {m['name']}  (tactic: {m['tactics']})")
        print(f"    {_first_sentence(m['description'])}")
        print(f"    Mitigations: {m['mitigations'][:300]}")
    else:
        print(f"\n[1] MITRE  no technique in lookup table; tactic: {', '.join(result['tactics']) or 'unknown'}")
    print(f"\n[2] CVE  (mode: {result['cve_mode'] or 'skipped'})")
    print("    skipped (no product, or technique is not exploit-type)" if result["cve"] is None
          else result["cve"].to_string(index=False) if len(result["cve"]) else "    no CVE matched")
    print("\n[3] CTI")
    print(result["cti"].to_string(index=False) if len(result["cti"]) else "    no CTI chunk for this technique or tactic")
    print("\n" + "=" * 100 + "\n")

In [ ]:
DEMO_EVENTS = [
    {"technique_id": "T1190", "service": "http", "port": 80, "protocol": "tcp",
     "behavior": "repeated crafted HTTP requests from an external IP to the public web server",
     "product": "Apache HTTP Server", "version": "2.4"},
    {"technique_id": "T1071", "service": "ssl", "port": 443, "protocol": "tcp",
     "behavior": "periodic outbound connections from an internal host to a rare external IP"},
    {"technique_id": "T1021", "service": "smb", "port": 445, "protocol": "tcp",
     "behavior": "one internal host opening SMB sessions to many other internal hosts"},
]
for ev in DEMO_EVENTS:
    show(retrieve(ev))

In [ ]:
# Why the technique filter matters: same T1190 event, CTI results with vs. without the filter
ev = DEMO_EVENTS[0]
print("CTI without filter:\n", retrieve(ev, use_filter=False)["cti"].to_string(index=False))
print("\nCTI with technique filter:\n", retrieve(ev)["cti"].to_string(index=False))

## 6. Golden dataset + baseline retrieval evaluation
- Ground truth = CTI-HAL's human labels: for a technique query, every CTI chunk labeled with that technique is relevant.
- **Technique queries:** techniques with at least `MIN_CHUNKS` CTI chunks; query = MITRE name + first sentence of its
  ATT&CK description (written independently of CTI-HAL).
- **Network event queries:** hand-written UWF-style event descriptions.
- Evaluated on **unfiltered** vector search (with the technique filter, precision is 1 by construction).
- Metrics: Hit@5, Precision@5, MRR, nDCG@10.

In [ ]:
import math
from collections import defaultdict

MIN_CHUNKS = 5

relevant = defaultdict(list)
for c in cti_chunks:
    for t in c["metadata"]["technique_ids"]:
        relevant[t].append(c["id"])

golden = []
for tid, ids in sorted(relevant.items(), key=lambda x: -len(x[1])):
    m = _mitre_for(tid)
    if len(ids) < MIN_CHUNKS or m is None:
        continue
    golden.append({"query_id": f"tech-{tid}", "technique_id": tid, "query_type": "technique",
                   "query": f"{m['name']}. {_first_sentence(m['description'])}",
                   "relevant_ids": ids})

EVENT_QUERIES = [
    ("T1190", "External IP sends crafted HTTP requests to a public web server, followed by an unexpected outbound connection from the server"),
    ("T1071", "Internal host makes periodic HTTP/HTTPS connections to the same rare external IP at regular intervals"),
    ("T1021", "Internal host logs in to many other internal hosts over SMB and RDP within a short time window"),
    ("T1046", "Single host connects to many ports on many internal hosts with very short, mostly rejected connections"),
]
for tid, q in EVENT_QUERIES:
    if relevant.get(tid):
        golden.append({"query_id": f"event-{tid}", "technique_id": tid, "query_type": "network_event",
                       "query": q, "relevant_ids": relevant[tid]})

write_jsonl(golden, f"{OUT_DIR}/golden_cti.jsonl")
print(f"{len(golden)} queries "
      f"({sum(g['query_type'] == 'technique' for g in golden)} technique, "
      f"{sum(g['query_type'] == 'network_event' for g in golden)} network event)")
pd.DataFrame([{**g, "n_relevant": len(g["relevant_ids"])} for g in golden]).drop(columns="relevant_ids").head(10)

In [ ]:
def evaluate(golden, k=5, n=10):
    res = cti_col.query(query_embeddings=embed([g["query"] for g in golden]), n_results=n)
    rows = []
    for g, ids in zip(golden, res["ids"]):
        rel = set(g["relevant_ids"])
        hits = [i in rel for i in ids]
        dcg = sum(h / math.log2(r + 2) for r, h in enumerate(hits))
        idcg = sum(1 / math.log2(r + 2) for r in range(min(len(rel), n)))
        rows.append({
            "query_id": g["query_id"], "query_type": g["query_type"], "technique_id": g["technique_id"],
            "n_relevant": len(rel),
            f"hit@{k}": float(any(hits[:k])),
            f"precision@{k}": sum(hits[:k]) / k,
            "mrr": next((1 / (r + 1) for r, h in enumerate(hits) if h), 0.0),
            f"ndcg@{n}": dcg / idcg if idcg else 0.0,
        })
    return pd.DataFrame(rows)


eval_df = evaluate(golden)
eval_df.to_csv(f"{OUT_DIR}/eval_cti_baseline.csv", index=False)
metric_cols = ["hit@5", "precision@5", "mrr", "ndcg@10"]
print("Baseline (unfiltered vector search, all-MiniLM-L6-v2):\n")
print(eval_df.groupby("query_type")[metric_cols].mean().round(3).to_string())
print("\nOverall:", eval_df[metric_cols].mean().round(3).to_dict())
print("\nNetwork event queries:\n", eval_df[eval_df.query_type == "network_event"][["technique_id", "n_relevant"] + metric_cols].round(3).to_string(index=False))

## 7. UWF integration – connect the GNN output and choose the retrieval strategy
Inputs from the preprocessing notebook (`3_UWF_Windowed_Graph_Preprocessing`): `edges_win_15min.parquet` (one row per
src → dst edge in a 15-min window, with `y`, `tech_mode`, `tactic_mode` and aggregate features) and `node_static.csv`.

| Step | What it does |
|---|---|
| 7.1 | Load UWF edges (falls back to the published technique counts if the parquet is not reachable) |
| 7.2 | ATT&CK procedure examples (`uses` relationships) as a second evidence source → collection `procedures` |
| 7.3 | CTID KEV→ATT&CK mappings → `CTID_CVES` (technique → CVE IDs) for technique-gated CVE search |
| 7.4 | `edge_to_event()` – aggregated edge → the event dict `retrieve()` expects |
| 7.5 | **Decision table** – share of UWF attack edges that get evidence under each strategy |
| 7.6 | UWF golden queries + evaluation (unfiltered, comparable with section 6) |
| 7.7 | Retrieval on real UWF edges, one per main technique |
| 7.8 | Save the index (now with `procedures`) back to Drive |

In [ ]:
# 7.1 Load UWF edges

UWF_DIR = os.environ.get("UWF_GRAPH_DIR", "/content/drive/Shareddrives/298B/Phase B Preprocessing/uwf_graph_v2")
UWF_EDGES = f"{UWF_DIR}/edges_win_15min.parquet"
UWF_NODES = f"{UWF_DIR}/node_static.csv"

# Fallback: attack edges per tech_mode (5-min windows), copied from the preprocessing notebook output
UWF_TECH_EDGES_FALLBACK = {
    "T1595": 36761, "T1110": 5186, "T1078": 4239, "T1190": 3696, "T1046": 542,
    "Reconnaissance_unspecified": 290, "T1018": 273, "T1548": 264, "T1592": 230, "T1048": 119,
    "T1587": 23, "T1589": 18, "T1210": 17, "T1016": 13, "Credential Access_unspecified": 6,
    "T1071": 4, "T1590": 3, "Privilege Escalation_unspecified": 3, "Exfiltration_unspecified": 3, "T1059": 3,
}

if os.path.exists(UWF_EDGES):
    uwf_edges = pd.read_parquet(UWF_EDGES)
    uwf_att = uwf_edges[uwf_edges.y == 1].copy()
    uwf_tech = uwf_att.tech_mode.value_counts()
    print(f"UWF edges: {len(uwf_edges):,} | attack edges: {len(uwf_att):,} | techniques: {uwf_tech.size}")
else:
    uwf_edges = uwf_att = None
    uwf_tech = pd.Series(UWF_TECH_EDGES_FALLBACK).sort_values(ascending=False)
    print(f"{UWF_EDGES} not found -> using fallback counts (top {uwf_tech.size} techniques, "
          f"{uwf_tech.sum():,} of 51,709 attack edges). 7.4/7.7 need the parquet.")

uwf_ip = (pd.read_csv(UWF_NODES).set_index("node_id") if os.path.exists(UWF_NODES) else None)

# UWF tactic per technique label (from the data when available, else from ATT&CK)
if uwf_att is not None and "tactic_mode" in uwf_att:
    UWF_TACTIC = uwf_att.groupby("tech_mode").tactic_mode.agg(lambda s: s.mode().iat[0] if s.notna().any() else None).to_dict()
else:
    UWF_TACTIC = {}
uwf_tech.head(10)

In [ ]:
import pandas as pd
e = pd.read_parquet("/content/drive/Shareddrives/298B/Phase B Preprocessing/uwf_graph_v2/edges_win_15min.parquet")
need = ["win_id","src_id","dst_id","y","tech_mode","tactic_mode","n_logs","n_dst_ports","avg_log_orig_bytes",
        "total_http","total_dns","total_ssl","total_ssh","total_tcp","total_udp","total_icmp",
        "f_cs_sf","f_cs_s0","f_cs_rej","f_cs_rst","f_pg_web","f_pg_ssh","f_pg_smb_rdp","f_pg_dns",
        "f_pg_other_wk","f_pg_registered","f_pg_ephemeral"]
print(e.shape, "| 缺少的列：", [c for c in need if c not in e.columns])

In [ ]:
# 7.2 ATT&CK procedure examples: "uses" relationships (group/software/campaign -> technique) with a description.
# Same source file as section 1, so no extra download. Fills gaps where CTI-HAL has no chunk (e.g. T1595).
def _clean_attack_text(text):
    text = re.sub(r"\(Citation:[^)]*\)", "", text or "")
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)     # [name](url) -> name
    return re.sub(r"\s+", " ", text).strip()


def parse_procedures(path):
    with open(path, encoding="utf-8") as f:
        objects = json.load(f)["objects"]
    by_id = {o["id"]: o for o in objects}
    chunks, stats = [], Counter()
    for o in objects:
        if o.get("type") != "relationship" or o.get("relationship_type") != "uses" or not _active(o):
            continue
        tgt, srco = by_id.get(o.get("target_ref")), by_id.get(o.get("source_ref"))
        if not tgt or tgt.get("type") != "attack-pattern" or not _active(tgt) or not srco or not _active(srco):
            continue
        tid, _ = _attack_id(tgt)
        if not tid:
            continue
        text = _clean_attack_text(o.get("description"))
        if len(text.split()) < 4:
            stats["dropped_too_short"] += 1
            continue
        src_id, src_url = _attack_id(srco)
        chunks.append({
            "id": "proc-" + o["id"].split("--")[-1],
            "text": text,
            "metadata": {
                "source": "attack-procedure",
                "technique_id": tid,
                "technique_ids": sorted({tid, tid.split(".")[0]}),   # filter on parent or sub-technique
                "source_name": srco.get("name", ""),
                "source_type": srco.get("type", ""),                 # intrusion-set / malware / tool / campaign
                "source_attack_id": src_id or "",
                "url": src_url or "",
            },
        })
    stats["kept"] = len(chunks)
    print(dict(stats))
    return chunks


proc_chunks = parse_procedures(ATTACK_JSON)
write_jsonl(proc_chunks, f"{OUT_DIR}/procedure_chunks.jsonl")
proc_col = build_collection(client, "procedures", proc_chunks, embed)

proc_labels = pd.DataFrame([c["metadata"] for c in proc_chunks]).explode("technique_ids")
proc_count = proc_labels[~proc_labels.technique_ids.str.contains(r"\.")].technique_ids.value_counts()
print("\nProcedure examples for UWF techniques:\n",
      proc_count.reindex(UWF_TECHNIQUES, fill_value=0).to_string())

In [ ]:
# 7.3 CTID KEV -> ATT&CK mappings (Mappings Explorer, https://ctid.mitre.org/projects/mappings-explorer/).
# Download the KEV mapping file (JSON or CSV) once and put it on Drive; the shell in Colab can also fetch it.
# Each mapping: capability_id = CVE ID, attack_object_id = technique, mapping_type = exploitation_technique /
# primary_impact / secondary_impact. Only exploitation_technique answers "which CVE could this technique exploit".
CTID_FILE = os.environ.get("RAG_CTID_FILE", f"{DRIVE_ROOT}/ctid/kev_attack_mappings.json")   # edit file name
CTID_TYPES = {"exploitation_technique"}


def load_ctid(path, types=CTID_TYPES):
    if path.endswith(".csv"):
        objs = pd.read_csv(path).to_dict("records")
    else:
        data = json.load(open(path, encoding="utf-8"))
        objs = data.get("mapping_objects", []) if isinstance(data, dict) else data
    m, seen_types = defaultdict(set), Counter()
    for r in objs:
        cve, tech, typ = str(r.get("capability_id", "")).upper(), str(r.get("attack_object_id", "")), str(r.get("mapping_type", ""))
        seen_types[typ] += 1
        if cve.startswith("CVE-") and re.match(r"T\d{4}", tech) and (not typ or typ in types):
            m[tech.split(".")[0]].add(cve)
    print("mapping types in file:", dict(seen_types))
    return m


corpus_cves = {c["metadata"]["cve_id"] for c in read_jsonl(f"{OUT_DIR}/cve_chunks.jsonl")}
CTID_CVES = {}
if os.path.exists(CTID_FILE):
    raw = load_ctid(CTID_FILE)
    CTID_CVES = {t: ids & corpus_cves for t, ids in raw.items() if ids & corpus_cves}
    print(f"{sum(len(v) for v in raw.values())} technique-CVE links, "
          f"{len(set().union(*raw.values())) if raw else 0} CVEs; "
          f"{len(set().union(*CTID_CVES.values())) if CTID_CVES else 0} of them are in our network CVE corpus")
    print("CVEs per exploit-type technique (in corpus):",
          {t: len(CTID_CVES.get(t, ())) for t in sorted(CVE_TECHNIQUES)})
else:
    print(f"{CTID_FILE} not found -> CVE_MODE 'auto' uses the 'service' fallback")

In [ ]:
# 7.4 edge_to_event(): one aggregated UWF edge (src -> dst, one 15-min window) -> event dict for retrieve()
# DRAFT – the field mapping still needs to be agreed with the GNN team (Vedika / Sushma).
SERVICE_COLS = {"http": "total_http", "dns": "total_dns", "ssl": "total_ssl", "ssh": "total_ssh"}
PROTO_COLS = {"tcp": "total_tcp", "udp": "total_udp", "icmp": "total_icmp"}
PORT_GROUPS = {   # name: (column, label used in the query, service hint when Zeek's service is unknown)
    "web": ("f_pg_web", "80/443", "http"), "ssh": ("f_pg_ssh", "22", "ssh"),
    "smb_rdp": ("f_pg_smb_rdp", "445/3389", "smb"), "dns": ("f_pg_dns", "53", "dns"),
    "other_wk": ("f_pg_other_wk", "other well-known (<1024)", None),
    "registered": ("f_pg_registered", "registered (1024-49151)", None),
    "ephemeral": ("f_pg_ephemeral", "ephemeral (>=49152)", None),
}


def _val(x):
    return None if x is None or (isinstance(x, float) and math.isnan(x)) else x


def _argmax(row, cols):
    k, v = max(((k, float(row.get(c, 0) or 0)) for k, c in cols.items()), key=lambda x: x[1])
    return k if v > 0 else None


def _host(node_id):
    if uwf_ip is None or node_id not in uwf_ip.index:
        return "host", f"node {node_id}"
    r = uwf_ip.loc[node_id]
    return ("internal host" if r.is_internal else "external host"), r.ip


def edge_to_event(row, technique_id=None, tactic=None):
    """row: one edges_win row. technique_id / tactic: the model's prediction (default: ground-truth tech_mode)."""
    tid = technique_id or _val(row.get("tech_mode"))
    service = _argmax(row, SERVICE_COLS) or "unknown"
    proto = _argmax(row, PROTO_COLS) or "unknown"
    pg = _argmax(row, {k: v[0] for k, v in PORT_GROUPS.items()})
    pg_label, hint = (PORT_GROUPS[pg][1], PORT_GROUPS[pg][2]) if pg else ("unknown", None)
    n, ports = int(row.n_logs), int(row.n_dst_ports)
    fail = float(row.f_cs_s0 + row.f_cs_rej + row.f_cs_rst)
    sent = math.expm1(float(row.avg_log_orig_bytes))          # avg of ln(1+bytes) -> typical bytes sent
    src_kind, src_ip = _host(int(row.src_id))
    dst_kind, dst_ip = _host(int(row.dst_id))
    behavior = (f"{src_kind} made {n} {proto.upper()} connection{'s' if n != 1 else ''} to one {dst_kind} within 15 minutes, "
                f"{ports} distinct destination port{'s' if ports != 1 else ''} (mostly {pg_label}); "
                f"{fail:.0%} failed or rejected, {float(row.f_cs_sf):.0%} completed normally; "
                f"about {sent:,.0f} bytes sent per connection")
    return {"technique_id": tid, "tactic": tactic or _val(row.get("tactic_mode")) or UWF_TACTIC.get(tid),
            "service": service if service != "unknown" else (hint or "unknown"), "service_hint": hint,
            "port": pg_label, "protocol": proto, "behavior": behavior,
            "src_ip": src_ip, "dst_ip": dst_ip, "win_id": int(row.win_id)}


def representative_edges(att, n_per_tech=1, seed=0):
    """Edges closest to the median n_logs of each technique (typical, not extreme)."""
    out = []
    for t, g in att.groupby("tech_mode"):
        med = g.n_logs.median()
        out.append(g.iloc[(g.n_logs - med).abs().argsort()[:n_per_tech]])
    return pd.concat(out)


if uwf_att is not None:
    rep = representative_edges(uwf_att)
    for t in ["T1595", "T1110", "T1190"]:
        if t in set(rep.tech_mode):
            print(json.dumps(edge_to_event(rep[rep.tech_mode == t].iloc[0]), indent=1, default=str), "\n")

In [ ]:
INTERNAL_IP = re.compile(r"^(10\.|192\.168\.|172\.(1[6-9]|2[0-9]|3[01])\.|143\.88\.)")
HOST_ONLY = {"T1204", "T1547", "T1136", "T1059", "T1112", "T1546"}   # not observable from Zeek conn logs


def _kind(ip):
    return "internal host" if INTERNAL_IP.match(ip or "") else "external host"


def gnn_output_to_events(out, window_min=5, margin=0.05, max_k=2):
    """Sai's GNN inference JSON -> retrieve() events.
    Technique scores are independent sigmoids that sit close together, so keep the top-1 plus any
    candidate within `margin` of it (max `max_k`), after dropping host-only techniques."""
    det = out["gnn_detection"]
    if det["binary_prediction"]["label"] != "attack":
        return []
    ev, f = out["event"], out["event"]["traffic_features"]

    protos = {"tcp": f.get("tcp_count", 0), "udp": f.get("udp_count", 0), "icmp": f.get("icmp_count", 0)}
    proto = max(protos, key=protos.get) if any(protos.values()) else "unknown"
    n = int(f["connection_count"])
    behavior = (f"{_kind(ev['source_ip'])} made {n:,} {proto.upper()} connection{'s' if n != 1 else ''} "
                f"to one {_kind(ev['destination_ip'])} within {window_min} minutes; "
                f"{f['total_orig_bytes']:,.0f} bytes sent and {f['total_resp_bytes']:,.0f} bytes received "
                f"({f['total_orig_packets']:,.0f}/{f['total_resp_packets']:,.0f} packets), "
                f"total duration {f['total_duration']:.2f}s")
    cs = f.get("conn_state_ratios") or {}
    if cs:                                   # used automatically once Sai fills these fields
        fail = sum(cs.get(k, 0) for k in ("S0", "REJ", "RSTO", "RSTR", "RSTOS0", "RSTRH"))
        behavior += f"; {fail:.0%} failed or rejected, {cs.get('SF', 0):.0%} completed normally"
    service = f.get("dominant_zeek_service") or "unknown"
    port = f.get("destination_port_group") or f.get("dominant_destination_port") or "unknown"

    ranked = sorted(det["mitre_candidates"], key=lambda c: -c["confidence"])
    cands = [c for c in ranked if c["technique_id"].split(".")[0] not in HOST_ONLY] or ranked[:1]
    top = cands[0]["confidence"]
    kept = [c for c in cands if top - c["confidence"] <= margin][:max_k]

    tactic = (det.get("tactic_prediction") or {}).get("label")
    return [{"technique_id": c["technique_id"], "technique_confidence": c["confidence"],
             "tactic": None,                 # tactic head not reliable yet; let MITRE give the tactic
             "service": service, "port": str(port), "protocol": proto, "behavior": behavior,
             "src_ip": ev["source_ip"], "dst_ip": ev["destination_ip"],
             "event_id": out["event_id"], "gnn_tactic": tactic} for c in kept]

In [ ]:
client = chromadb.PersistentClient(path=local_chroma)
cve_col, cti_col, proc_col = (client.get_collection(n) for n in ["cve", "cti", "procedures"])
print({c.name: c.count() for c in [cve_col, cti_col, proc_col]})
sai_sample = {
  "event_id": "uwf-test-231530", "row_id": 231530,
  "event": {"window_start": "2025-07-16T02:20:00", "source_ip": "143.88.1.13", "destination_ip": "143.88.1.11",
            "traffic_features": {"connection_count": 30, "total_duration": 0.4237, "maximum_duration": 0.1716,
                                 "total_orig_bytes": 74667.0, "total_resp_bytes": 1442105.0,
                                 "total_orig_packets": 413.0, "total_resp_packets": 1066.0,
                                 "tcp_count": 30.0, "udp_count": 0.0, "icmp_count": 0.0}},
  "gnn_detection": {
    "binary_prediction": {"label": "attack", "attack_probability": 0.9949, "confidence": 0.9949},
    "selected_mitre_id": "T1592",
    "mitre_candidates": [{"rank": 1, "technique_id": "T1592", "confidence": 0.7184},
                         {"rank": 2, "technique_id": "T1590", "confidence": 0.6943},
                         {"rank": 3, "technique_id": "T1204", "confidence": 0.5537}],
    "tactic_prediction": {"label": "Reconnaissance", "confidence": 0.6759}}}

events = gnn_output_to_events(sai_sample)
print("kept candidates:", [e["technique_id"] for e in events])
for e in events:
    show(retrieve(e))

In [ ]:
# 7.5 Decision table: how much of UWF's attack traffic gets evidence under each strategy (no embeddings needed)
#   S1  CTI-HAL, technique filter only (Demo-1)
#   S2  S1 + ATT&CK procedure examples for the same technique
#   S3  S2 + CTI-HAL chunks of other techniques in the same tactic
K_EVIDENCE = 5
cti_count = labels.technique_ids.value_counts()                      # from section 3 (parent IDs)
cti_by_parent = defaultdict(set)
for c in cti_chunks:
    for t in c["metadata"]["technique_ids"]:
        cti_by_parent[t].add(c["id"])

rows = []
for label, n_edges in uwf_tech.items():
    tid, tactic = _resolve_ids({"technique_id": label, "tactic": UWF_TACTIC.get(label)})
    parent = tid.split(".")[0] if tid else None
    m = _mitre_for(tid) if tid else None
    tactics = [tactic] if tactic else (m["tactics"].split(", ") if m else [])
    n_cti = int(cti_count.get(parent, 0)) if parent else 0
    n_proc = int(proc_count.get(parent, 0)) if parent else 0
    sib = {t for ta in tactics for t in _parents_for_tactic(ta)} - {parent}
    n_tac = len(set().union(*(cti_by_parent[t] for t in sib)) - cti_by_parent.get(parent, set())) if sib else 0
    ctid = len(CTID_CVES.get(parent, ())) if parent else 0
    rows.append({"technique": label, "name": m["name"] if m else "-", "tactic": ", ".join(tactics),
                 "attack_edges": int(n_edges), "cti": n_cti, "procedures": n_proc, "tactic_cti": n_tac,
                 "S1": n_cti, "S2": n_cti + n_proc, "S3": n_cti + n_proc + n_tac,
                 "cve_service": parent in CVE_TECHNIQUES, "cve_ctid": ctid})
cov = pd.DataFrame(rows)
cov.to_csv(f"{OUT_DIR}/uwf_coverage.csv", index=False)
print(cov.drop(columns=["S1", "S2", "S3"]).to_string(index=False))

tot = cov.attack_edges.sum()
summary = pd.DataFrame([{
    "strategy": s,
    "edges with >=1 evidence": f"{cov.loc[cov[s] >= 1, 'attack_edges'].sum() / tot:.1%}",
    f"edges with >={K_EVIDENCE}": f"{cov.loc[cov[s] >= K_EVIDENCE, 'attack_edges'].sum() / tot:.1%}",
    "techniques with >=1": f"{(cov[s] >= 1).sum()}/{len(cov)}",
} for s in ["S1", "S2", "S3"]])
print("\nCTI strategies (weighted by UWF attack edges):\n", summary.to_string(index=False))

cve_rows = [
    {"CVE mode": "product", "edges where CVE search runs": "0.0%  (UWF conn logs have no product field)"},
    {"CVE mode": "service", "edges where CVE search runs": f"{cov.loc[cov.cve_service, 'attack_edges'].sum() / tot:.1%}"},
    {"CVE mode": "ctid", "edges where CVE search runs":
        f"{cov.loc[cov.cve_ctid > 0, 'attack_edges'].sum() / tot:.1%}" if CTID_CVES else "n/a (no CTID file loaded)"},
]
print("\nCVE trigger coverage:\n", pd.DataFrame(cve_rows).to_string(index=False))

In [ ]:
# 7.6 UWF golden queries: one typical UWF edge per technique that has CTI-HAL chunks, written by edge_to_event()
# (no technique name in the query, like the hand-written network event queries in section 6).
# Ground truth = CTI-HAL chunks with that technique (exact) – and, as a looser check, any technique of the same tactic.
UWF_EVENT_QUERIES_FALLBACK = [   # used only if the UWF parquet is not reachable
    ("T1110", "Internal host made hundreds of TCP connections to one internal host within 15 minutes on a single port; all completed normally with small payloads"),
    ("T1078", "Internal host made a single TCP connection attempt to port 445 on another internal host that got no response"),
    ("T1190", "Internal host sent a completed TCP connection with a few hundred bytes to a server port on another internal host"),
    ("T1046", "One host connected to many destination ports on another host within 15 minutes; most connections were rejected or unanswered"),
    ("T1210", "Host connected to an SMB/RDP service on another internal host and exchanged data after the handshake"),
]

uwf_golden = []
if uwf_att is not None:
    for _, r in representative_edges(uwf_att).iterrows():
        tid, _ = _resolve_ids({"technique_id": r.tech_mode})
        if tid and relevant.get(tid.split(".")[0]):
            ev = edge_to_event(r)
            uwf_golden.append({"query_id": f"uwf-{tid}", "technique_id": tid, "query_type": "uwf_edge",
                               "query": f"{ev['behavior']} over {ev['service']}/{ev['protocol']} port {ev['port']}",
                               "relevant_ids": relevant[tid.split(".")[0]]})
else:
    for tid, q in UWF_EVENT_QUERIES_FALLBACK:
        if relevant.get(tid):
            uwf_golden.append({"query_id": f"uwf-{tid}", "technique_id": tid, "query_type": "uwf_handwritten",
                               "query": q, "relevant_ids": relevant[tid]})

write_jsonl(uwf_golden, f"{OUT_DIR}/golden_cti_uwf.jsonl")
print(f"{len(uwf_golden)} UWF queries for techniques with CTI chunks: {[g['technique_id'] for g in uwf_golden]}")
print("Techniques with UWF attack edges but no CTI-HAL chunk (cannot be scored):",
      [t for t in uwf_tech.index if (_resolve_ids({'technique_id': t})[0] or '').split('.')[0] not in relevant])

if uwf_golden:
    eval_uwf = evaluate(uwf_golden)
    # looser relevance: any CTI chunk whose technique shares a tactic with the query's technique
    res = cti_col.query(query_embeddings=embed([g["query"] for g in uwf_golden]), n_results=5)
    chunk_tech = {c["id"]: c["metadata"]["technique_ids"] for c in cti_chunks}
    tac_hit = []
    for g, ids in zip(uwf_golden, res["ids"]):
        sib = {t for ta in _mitre_for(g["technique_id"])["tactics"].split(", ") for t in _parents_for_tactic(ta)}
        tac_hit.append(float(any(set(chunk_tech[i]) & sib for i in ids)))
    eval_uwf["tactic_hit@5"] = tac_hit
    eval_uwf.to_csv(f"{OUT_DIR}/eval_cti_uwf.csv", index=False)
    print("\nUWF queries, unfiltered vector search (compare with section 6):\n",
          eval_uwf[["technique_id", "n_relevant"] + metric_cols + ["tactic_hit@5"]].round(3).to_string(index=False))
    print("\nMean:", eval_uwf[metric_cols + ["tactic_hit@5"]].mean().round(3).to_dict())

In [ ]:
# 7.7 Retrieval on real UWF edges (one typical edge per main technique) with the full strategy
if uwf_att is not None:
    main = [t for t in uwf_tech.index[:8]]
    rep = representative_edges(uwf_att[uwf_att.tech_mode.isin(main)])
    level_rows = []
    for _, r in rep.iterrows():
        res = retrieve(edge_to_event(r))
        show(res)
        lv = res["cti"].evidence_level.value_counts().to_dict() if len(res["cti"]) else {}
        level_rows.append({"uwf_label": r.tech_mode, "cve_mode": res["cve_mode"] or "-",
                           "n_cve": 0 if res["cve"] is None else len(res["cve"]), **lv})
    print("Evidence levels per technique (top 5 CTI results):\n",
          pd.DataFrame(level_rows).fillna(0).to_string(index=False))
else:
    print("UWF parquet not available – set UWF_DIR in 7.1")

In [ ]:
# 7.8 Save the updated index (cve + cti + procedures) back to Drive
drive_chroma = f"{OUT_DIR}/chroma_db"
del client, cve_col, cti_col, proc_col       # release the SQLite file before copying
shutil.rmtree(drive_chroma, ignore_errors=True)
shutil.copytree(local_chroma, drive_chroma)
print("Saved index to", drive_chroma)

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/Shareddrives/298A Group/Data/RAG Corpus/CVE_MITRE_Full_Scored_Dataset.csv")
print(df.shape)
print(df.columns.tolist())
df.head(3)